# Session 24: Preparing Data for Models

Data Science with Python, Level 2, Stage 4: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Explain why models need numeric, complete data.
2. Encode categories with OneHotEncoder.
3. Encode ordered categories with OrdinalEncoder.
4. Fill missing values with SimpleImputer.
5. Combine steps with ColumnTransformer.
6. Build one pipeline from raw table to prediction.

## Where We Are

Welcome to **Level 2** and Stage 4, Applied Machine Learning. In Stage 3 we prepared the churn data by hand: `(churn["contract"] == "Month-to-month").astype(int)` for every useful category. That does not scale, and it is easy to get wrong when new data arrives.

Today you learn scikit-learn's **transformers**. They encode text categories, fill missing values and scale numbers, and they remember what they learned from the training data so new rows are treated the same way. Throughout Stage 4 the cumulative project is a **customer churn model**.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
X = churn.drop(columns=["customer_id", "churn"])
y = (churn["churn"] == "Yes").astype(int)

## 1. What Is Wrong with Raw Data?

In [ ]:
print(X.dtypes)

In [ ]:
print(X.isna().sum()[X.isna().sum() > 0])

## 2. Encoding Categories

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(X[["contract"]])

print(encoder.get_feature_names_out())
print(encoded[:3])

In [ ]:
order = [["Month-to-month", "One year", "Two year"]]
ordinal = OrdinalEncoder(categories=order)
X_ord = ordinal.fit_transform(X[["contract"]])
print(pd.Series(X_ord.ravel()).value_counts().sort_index())

In [ ]:
new_customer = pd.DataFrame({"internet": ["Fiber", "Satellite"]})
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(X[["internet"]].fillna("None"))
print(enc.get_feature_names_out())
print(enc.transform(new_customer))

## 3. Filling Missing Values

In [ ]:
imputer = SimpleImputer(strategy="median")
imputer.fit(X[["total_charges"]])
print("Fill value:", imputer.statistics_)

filled = imputer.transform(X[["total_charges"]])
print("Missing after:", np.isnan(filled).sum())

| Strategy | Use for | Example |
| --- | --- | --- |
| median | Numbers with outliers | Income, charges |
| mean | Numbers without strong skew | Temperature |
| most_frequent | Categories | City, gender |
| constant | Missing has a meaning | No internet becomes "None" |

Missing values can carry information. Here a missing internet type means the customer has no internet.

## 4. ColumnTransformer

In [ ]:
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls"]
cat_cols = ["gender", "contract", "internet", "payment_method"]
bool_cols = ["senior", "paperless_billing"]

num_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
cat_steps = make_pipeline(SimpleImputer(strategy="constant", fill_value="None"),
                          OneHotEncoder(handle_unknown="ignore"))

In [ ]:
preprocess = ColumnTransformer([
    ("num", num_steps, num_cols),
    ("cat", cat_steps, cat_cols),
    ("bool", "passthrough", bool_cols),
])
X_ready = preprocess.fit_transform(X)
print(X_ready.shape)
print(preprocess.get_feature_names_out()[:7])

In [ ]:
model = make_pipeline(preprocess, LogisticRegression(max_iter=1000))
scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
print("AUC:", scores.mean().round(3))

In [ ]:
model.fit(X, y)
new = pd.DataFrame([{
    "gender": "Female", "senior": False, "tenure_months": 3,
    "contract": "Month-to-month", "internet": "Satellite",
    "monthly_charges": 95.0, "total_charges": np.nan,
    "support_calls": 4, "payment_method": "Electronic check",
    "paperless_billing": True}])
print("Risk of leaving:", model.predict_proba(new)[:, 1].round(2))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Leakage | Imputing or scaling the whole dataset before splitting | Put every step inside a pipeline. |
| Fake order | Using OrdinalEncoder for cities | Use one-hot for categories without a natural order. |
| Unknown crash | New category at prediction time raises an error | Use `handle_unknown="ignore"`. |
| Dropping too much | `dropna()` removing hundreds of rows | Impute, and keep the information. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Payment Dummies

One-hot encode `payment_method` with `OneHotEncoder(sparse_output=False)`. Put the result in a DataFrame with the feature names as columns and print the column sums.

In [ ]:
# Write your code below

### In-Class Exercise 2: Survey Gaps

Load `messy_survey.csv`. Count missing values per column, then fill `monthly_income` and `hours_online` with the median and `city` with the most frequent value using `SimpleImputer`. Check that no gaps remain in those columns.

In [ ]:
# Write your code below

### In-Class Exercise 3: Salary Preprocessor

Load `employees.csv`. Build a ColumnTransformer that one-hot encodes `department` and `city`, ordinal encodes `level` (Junior, Mid, Senior, Lead), and scales `years_experience` and `performance`. Print the shape of the transformed data.

In [ ]:
# Write your code below

### In-Class Exercise 4: Does It Help?

Compare the cross-validated AUC of the full churn pipeline with a version that drops `gender` and `paperless_billing`. Does removing them hurt?

In [ ]:
# Write your code below

## Summary

- Models need numeric data with no missing values.
- OneHotEncoder: one 0 or 1 column per category.
- OrdinalEncoder: only for categories with a real order.
- SimpleImputer learns a fill value from the training data.
- ColumnTransformer sends each column group to its own steps.
- A pipeline takes raw rows in and gives predictions out.

## What's Next

Linear and logistic regression draw straight boundaries. Next session you will meet decision trees, which ask a series of yes or no questions, and random forests, which combine hundreds of trees into one strong model.